# 🛠️ Aula 06 — Primeira Ferramenta
## Guilda de IA — Introdução à IA Generativa

<a href="https://colab.research.google.com/github/luksamuk/guilda-ia/blob/main/notebooks/aula06_primeira_ferramenta_colab.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

Ferramentas: dando *poderes* ao agente.

---

## 🏗️ Setup

⚠️ Vá em `Runtime → Change runtime type` → **T4 GPU**

Execute a célula abaixo e avance — é boilerplate reutilizável.

In [ ]:
# ── Setup completo: Ollama + gemma4:e2b + warm up ────────────────
# Tudo em uma célula. Execute e prossiga.

# 1. Instalar Ollama + deps Python
!apt-get install -y zstd pciutils lshw > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q langchain-openai langchain-core langchain requests

# 2. Workaround GPU Colab + keep alive
import os
os.environ["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"

# 3. Iniciar servidor
!pkill -f ollama 2> /dev/null; sleep 1
import subprocess
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env={**os.environ})

# 4. Aguardar servidor
import time, requests
for i in range(30):
    try:
        if requests.get("http://localhost:11434/api/tags", timeout=2).status_code == 200:
            break
    except:
        time.sleep(1)

# 5. Baixar modelo
!ollama pull gemma4:e2b

# 6. Warm up
print("🔥 Warm up...")
start = time.time()
!curl -s http://localhost:11434/api/chat -d '{"model":"gemma4:e2b","messages":[{"role":"user","content":"Oi"}],"stream":false,"keep_alive":-1}' > /dev/null
print(f"✅ Pronto em {time.time()-start:.1f}s")

## 1. Ferramentas com `@tool`

`@tool` transforma uma função Python em ferramenta pro LLM. **Pydantic** faz parsing e validação automático — sem `eval`, sem AST.

In [ ]:
from langchain_core.tools import tool
from datetime import datetime

@tool
def calcular(a: float, b: float, operacao: str) -> float:
    """Realiza uma operação matemática entre dois números.
    Use quando precisar fazer cálculos numéricos.
    operacao: 'soma', 'subtracao', 'multiplicacao', 'divisao'
    """
    ops = {'soma': a+b, 'subtracao': a-b, 'multiplicacao': a*b, 'divisao': a/b if b != 0 else 'Erro: divisão por zero'}
    return ops.get(operacao, f"Erro: operação '{operacao}' não suportada")

@tool
def obter_horario() -> str:
    """Retorna a data e hora atual. Use quando o usuário perguntar sobre data ou hora."""
    return datetime.now().strftime('%d/%m/%Y %H:%M:%S')

print("✅ Ferramentas criadas:", [calcular.name, obter_horario.name])

## 2. Ciclo manual: `bind_tools`

`llm.bind_tools()` registra as ferramentas. O LLM **decide** qual chamar, mas **não executa** — isso fica por sua conta.

In [ ]:
from pprint import pprint
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gemma4:e2b",
    base_url="http://localhost:11434/v1",
    api_key="nao_precisa",
    temperature=0,
)

llm_com_ferramentas = llm.bind_tools([calcular, obter_horario])

# Passo 1: O LLM decide, mas NÃO executa
resposta = llm_com_ferramentas.invoke("Quanto é 234 vezes 987?")
print("Content:", repr(resposta.content))  # string vazia!
print("\nTool calls:")
pprint(resposta.tool_calls)

In [ ]:
# Passo 2: Executar a ferramenta
tc = resposta.tool_calls[0]
if tc['name'] == 'calcular':
    resultado = calcular.invoke(tc['args'])
elif tc['name'] == 'obter_horario':
    resultado = obter_horario.invoke(tc['args'])
print(f"Resultado da ferramenta: {resultado}")  # 230958.0

In [ ]:
# Passo 3: Resultado volta pro LLM
from langchain_core.messages import HumanMessage, ToolMessage

mensagens = [
    HumanMessage(content="Quanto é 234 vezes 987?"),
    resposta,  # AIMessage com tool_call
    ToolMessage(content=str(resultado), tool_call_id=tc['id']),
]

resposta_final = llm_com_ferramentas.invoke(mensagens)
print(resposta_final.content)

## 3. Ciclo automático: `create_agent`

O ciclo manual é bom pra aprender. Na prática, `create_agent` faz tudo automaticamente.

In [ ]:
from langchain.agents import create_agent

agente = create_agent(llm, [calcular, obter_horario])

# Uma linha — decidir, executar, responder
resultado = agente.invoke({"input": "Quanto é 234 vezes 987?"})
print(resultado["output"])

In [ ]:
# Testar com horário e pergunta sem ferramenta
print(agente.invoke({"input": "Que horas são?"})["output"])
print()
print(agente.invoke({"input": "Qual a capital da França?"})["output"])

## 4. Um gostinho de *async*

Todas as chamadas até agora usam `invoke()` — síncrono, o código espera.
Mas existe `ainvoke()` (com **a** de async) que não bloqueia:

In [ ]:
# Síncrono (o que vimos)
# resposta = agente.invoke({"input": "Quanto é 234 vezes 987?"})

# Assíncrono — não bloqueia enquanto espera
# resposta = await agente.ainvoke({"input": "Quanto é 234 vezes 987?"})

# Próxima aula vamos usar isso com MCP!

---
✅ **Resumo:**
- **`@tool`** transforma funções Python em ferramentas (Pydantic faz parsing)
- **`bind_tools`** → ciclo manual (entender o que acontece)
- **`create_agent`** → ciclo automático (produzir)
- O LLM *decide* qual ferramenta usar — você só registra